# Home Credit · Loan amount with tabular foundation models
### TabTune on the Lexsi platform (SDK) · regression

Predict the **credit amount** a retail applicant should be offered (`AMT_CREDIT`) from their profile, bureau history and previous applications — a limit-setting and affordability use case.

| Step | What you will see |
|---|---|
| 1 | Connect, create a regression project |
| 2 | Data dashboard for the target and its drivers |
| 3 | **Leakage guard**: drop fields that restate the answer |
| 4 | Train several foundation models zero-shot, with SHAP and LIME |
| 5 | Model comparison from the platform's metrics |
| 6 | Explainability across models |
| 7 | Batch scoring on held-out applicants: predicted vs actual, error bands |
| 8 | One applicant, explained |
| 9 | Ensemble, distillation to a fast student, leaderboard |

**Before you start:** the processed Home Credit parquet files must be in Google Drive at `MyDrive/datasets/home-credit-default-risk/processed/`, and you need a Lexsi SDK access token.

## 1 · Connect

In [ ]:
# ⚙️ Install the Lexsi SDK (pinned to the version the backend was tested with)
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lexsi-sdk==0.1.69"], check=True)
from importlib.metadata import version
print("lexsi-sdk", version("lexsi-sdk"))

In [ ]:
# 🔐 Authenticate (paste your Lexsi SDK access token when prompted)
from lexsi_sdk import lexsi
lexsi.login()

In [ ]:
# 🔧 Demo configuration — the only cell you should need to edit
import os
ORGANIZATION = "Lexsi Labs"
WORKSPACE    = "TabTunetesting"
PROJECT      = "TFM Demo · Home Credit regression"   # project_type is fixed per project, so regression gets its own
COMPUTE      = "T4.small"
UPLOAD_COMPUTE = "small"                      # CPU size for the default XGBoost build on first upload

DATA_DIR = os.environ.get("HC_DATA_DIR", "/content/drive/MyDrive/datasets/home-credit-default-risk/processed")
TARGET   = "AMT_CREDIT"
SAMPLE_TRAIN_N, SAMPLE_TEST_N = 4000, 1500          # the backend caps a build at 10,000 rows (train + test)

# Regression-capable roster. XRFM and TabICLv2 are commercially usable; Limix is research-licensed.
ROSTER = ["TabICLv2", "XRFM", "Limix", "TabDPT"]
RUN_SUPPORT_SET  = False    # support-set attribution is defined for classification only
RUN_ENSEMBLE     = True
RUN_DISTILLATION = True
RUN_LEADERBOARD  = True
LIME_ITERATIONS  = 200
TABPFN_TOKEN     = None     # set to add "TabPFNv3" to the roster

In [ ]:
# 🧰 Helpers: idempotent project setup, safe training, metric extraction, plotting
import time, json, re
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from contextlib import contextmanager
from IPython.display import display, Markdown

BLUE, ORANGE, TEAL, GREY, DARK, RED = "#2F5BEA", "#B4541A", "#1B998B", "#9AA0A8", "#16181D", "#D7263D"
pd.set_option("display.max_columns", 50)


def get_or_create(workspace, name, project_type):
    """Reuse a project if it exists, otherwise create it. Safe to rerun."""
    try:
        project = workspace.project(name)
        print(f"reusing project: {name}")
    except Exception:
        project = workspace.create_project(name, modality="tabular", project_type=project_type)
        print(f"created project: {name}")
    return project


def upload_once(project, csv_path, tag, config=None, compute_type=None):
    """Upload a file unless this tag already has it; set the config only on the first upload.

    Two lexsi-sdk 0.1.69 workarounds, applied to the first upload only:
      1. upload_data() deletes the freshly uploaded file -> delete_file is suspended.
      2. an empty pred_label is no longer sent, and the backend then fails with
         "'NoneType' object has no attribute 'replace'" -> pred_label="" is put back in the request.
    """
    try:
        files = project.files()
        existing = set(files["file_name"]) if "file_name" in files else set()
    except Exception:
        existing = set()
    name = csv_path.split("/")[-1]
    has_config = project.config() != "Not Found"
    if name in existing and (has_config or config is None):
        print(f"already uploaded: {name} ({tag})")
        return
    if config is None or has_config:
        print(project.upload_data(csv_path, tag))
        return

    if name in existing:                                   # left over from a failed first upload
        project.delete_file(name)
    cls, api_cls = type(project), type(project.api_client)
    had_own, real_delete = "delete_file" in cls.__dict__, cls.__dict__.get("delete_file")
    real_post = api_cls.post

    def post_with_pred_label(self, uri, payload=None, *a, **k):
        if isinstance(payload, dict) and "unique_identifier" in payload and payload.get("pred_label") is None:
            payload["pred_label"] = ""
        return real_post(self, uri, payload, *a, **k)

    cls.delete_file = lambda self, *a, **k: "skipped"
    api_cls.post = post_with_pred_label
    try:
        print(project.upload_data(csv_path, tag, config=config, compute_type=compute_type))
    finally:
        api_cls.post = real_post
        if had_own:
            cls.delete_file = real_delete
        else:
            del cls.delete_file


@contextmanager
def tolerant_drop():
    """project.models() drops columns that some backend builds do not return."""
    orig = pd.DataFrame.drop

    def safe(self, *a, **k):
        k.setdefault("errors", "ignore")
        return orig(self, *a, **k)

    pd.DataFrame.drop = safe
    try:
        yield
    finally:
        pd.DataFrame.drop = orig


RUNS = []          # one record per training call, shown in the run log


def trained_models(project):
    with tolerant_drop():
        df = project.models()
    return df


def summary_of(project, model_name):
    try:
        return project.model_summary(model_name=model_name)
    except Exception as exc:
        print(f"no summary for {model_name}: {exc}")
        return None


def gfi(summary, method):
    """Global feature importance dict {feature: value} for 'shap' or 'lime', or {}."""
    if summary is None:
        return {}
    block = (summary.model_results or {}).get("GFI", {}) or {}
    raw = block.get(f"{method}_gfi") or {}
    out = {}
    for k, v in raw.items():
        try:
            out[k] = float(np.mean(np.abs(list(v.values())))) if isinstance(v, dict) else float(v)
        except Exception:
            pass
    return out


def support_rows(summary):
    if summary is None:
        return pd.DataFrame()
    block = ((summary.model_results or {}).get("GFI", {}) or {}).get("support_set_attribution") or {}
    return pd.DataFrame(block.get("top_support_rows", []))


def top_bar(importance, title, n=15, color=BLUE):
    s = pd.Series(importance).abs().sort_values(ascending=False).head(n)[::-1]
    fig = go.Figure(go.Bar(x=s.values, y=s.index, orientation="h", marker_color=color))
    fig.update_layout(template="plotly_white", height=36 * len(s) + 140, title=title,
                      xaxis_title="mean |attribution|", margin=dict(l=10, r=10, t=60, b=40))
    return fig

In [ ]:
def model_names(project):
    try:
        return set(trained_models(project)["model_name"].astype(str))
    except Exception:
        return set()


def train(project, label, **kwargs):
    """train_model with timing, a status line, and the exact name of the model it created."""
    before, t0 = model_names(project), time.time()
    try:
        project.train_model(**kwargs)
        status = "ok"
    except Exception as exc:
        status = f"failed: {str(exc)[:160]}"
        xai = (kwargs.get("data_config") or {}).get("xai_method", [])
        if "support_set" in xai and "not allowed" in str(exc):
            print(f"↺ {label}: support set is not enabled for this account; retrying with SHAP and LIME")
            kwargs["data_config"] = {**kwargs["data_config"], "xai_method": [m for m in xai if m != "support_set"]}
            return train(project, label, **kwargs)
    new = sorted(model_names(project) - before)
    RUNS.append({"run": label, "strategy": kwargs.get("training_strategy"), "model": str(kwargs.get("model")),
                 "created": ", ".join(new), "status": status, "minutes": round((time.time() - t0) / 60, 1)})
    print(("✅ " if status == "ok" else "❌ ") + f"{label} → {', '.join(new) or '-'}: {status} ({RUNS[-1]['minutes']} min)")
    return new[-1] if (status == "ok" and new) else None


def base_data_config(xai):
    return {"tags": ["Training"], "test_tags": ["Test"], "feature_exclude": FEATURE_EXCLUDE,
            "xai_method": xai, "lime_explainability_iterations": LIME_ITERATIONS}


def model_config_for(models):
    models = [models] if isinstance(models, str) else list(models)
    return {"tabpfn_token": TABPFN_TOKEN} if "TabPFNv3" in models and TABPFN_TOKEN else None


def eval_metrics(summary):
    return dict((summary.model_results or {}).get("evalution_metrics", {}) or {}) if summary else {}


def modelling_info(summary):
    return dict((summary.model_results or {}).get("modelling_info", {}) or {}) if summary else {}


def features_used(summary):
    return list((summary.model_results or {}).get("feature_actual_used", []) or []) if summary else []


def ensure_inference(project, model_name):
    """Make `model_name` the active model and enable it for inference."""
    project.activate_model(model_name)
    try:
        project.update_inference_model_status(model_name, True)
    except Exception:
        pass


def case_attributions(case, kind="shap", positive="1"):
    """Signed per-feature attribution dict from a case, picking the positive class when nested."""
    raw = getattr(case, f"{kind}_feature_importance", {}) or {}
    if raw and isinstance(next(iter(raw.values())), dict):
        key = positive if positive in raw else list(raw)[-1]
        raw = raw[key]
    out = {}
    for k, v in raw.items():
        try:
            out[k] = float(v)
        except Exception:
            pass
    return out


def signed_bar(attr, title, n=12):
    s = pd.Series(attr)
    s = s.reindex(s.abs().sort_values(ascending=False).index).head(n)[::-1]
    fig = go.Figure(go.Bar(x=s.values, y=s.index, orientation="h",
                           marker_color=[RED if v > 0 else BLUE for v in s.values]))
    fig.update_layout(template="plotly_white", height=34 * len(s) + 150, title=title,
                      xaxis_title="contribution (red pushes the prediction up)", margin=dict(l=10, r=10, t=60, b=40))
    return fig

In [ ]:
# 📐 Read the platform's inference output and score it locally
from sklearn import metrics as skm


def prediction_columns(result, uploaded):
    """Columns the platform added to the uploaded tag (prediction, probabilities, ...)."""
    return [c for c in result.columns if c not in set(uploaded.columns)]


def positive_proba(result, added, positive="1"):
    """Best guess at the positive-class probability column among the added columns."""
    numeric = [c for c in added if pd.api.types.is_numeric_dtype(result[c])
               and result[c].between(0, 1).all() and result[c].nunique() > 2]
    ranked = sorted(numeric, key=lambda c: (str(positive) not in c.split("_")[-1] and not c.endswith(str(positive)),
                                            "prob" not in c.lower() and "score" not in c.lower()))
    return ranked[0] if ranked else None


def predicted_label(result, added):
    cands = [c for c in added if result[c].nunique() <= 20]
    named = [c for c in cands if "pred" in c.lower()]
    return (named or cands or [None])[0]

In [ ]:
def regression_dashboard(y_true, y_pred, title, unit=""):
    """Predicted vs actual, residuals, error by decile and residual distribution."""
    y, f = np.asarray(y_true, float), np.asarray(y_pred, float)
    res = y - f
    out = {"rmse": float(np.sqrt(np.mean(res ** 2))), "mae": float(np.mean(np.abs(res))),
           "r2": float(skm.r2_score(y, f)), "mape_%": float(np.mean(np.abs(res) / np.maximum(np.abs(y), 1e-9)) * 100)}
    dec = pd.qcut(y, 10, labels=False, duplicates="drop")
    by_dec = pd.DataFrame({"dec": dec, "ape": np.abs(res) / np.maximum(np.abs(y), 1e-9) * 100}).groupby("dec")["ape"].median()
    lo, hi = np.percentile(np.r_[y, f], [1, 99])
    fig = make_subplots(rows=2, cols=2, vertical_spacing=0.15,
                        subplot_titles=(f"Predicted vs actual · R² {out['r2']:.3f}", "Residuals vs prediction",
                                        "Median absolute % error by actual-value decile", "Residual distribution"))
    fig.add_trace(go.Scattergl(x=y, y=f, mode="markers", marker=dict(color=BLUE, size=5, opacity=0.45), name="loan"), 1, 1)
    fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], line=dict(color=DARK, dash="dot"), name="perfect"), 1, 1)
    fig.add_trace(go.Scattergl(x=f, y=res, mode="markers", marker=dict(color=ORANGE, size=5, opacity=0.45), showlegend=False), 1, 2)
    fig.add_hline(y=0, line_color=DARK, line_dash="dot", row=1, col=2)
    fig.add_trace(go.Bar(x=[f"D{int(d) + 1}" for d in by_dec.index], y=by_dec.values, marker_color=TEAL, showlegend=False), 2, 1)
    fig.add_trace(go.Histogram(x=res, nbinsx=50, marker_color=GREY, showlegend=False), 2, 2)
    fig.update_xaxes(title_text=f"actual {unit}", row=1, col=1); fig.update_yaxes(title_text=f"predicted {unit}", row=1, col=1)
    fig.update_xaxes(title_text=f"predicted {unit}", row=1, col=2); fig.update_yaxes(title_text="% error", row=2, col=1)
    fig.update_layout(template="plotly_white", height=760, title=title, legend=dict(orientation="h", y=-0.08))
    fig.show()
    return out

In [ ]:
def preflight_high_cardinality(df, cols, limit=0.95, digits=(4, 3, 2)):
    """The backend drops any column with more than 95% unique values (model.py:494-500), which also
    removes continuous features. Round those columns to fewer significant digits so they survive."""
    report = []
    for c in cols:
        if not pd.api.types.is_numeric_dtype(df[c]):
            continue
        ratio = df[c].nunique() / len(df)
        if ratio <= limit:
            continue
        fixed = None
        for d in digits:
            rounded = df[c].map(lambda v: float(f"{v:.{d}g}") if pd.notna(v) else v)
            if rounded.nunique() / len(df) <= limit:
                fixed = d
                break
        report.append({"feature": c, "unique_ratio": round(ratio, 3),
                       "fix": f"rounded to {fixed} significant digits" if fixed else "dropped by backend"})
        if fixed:
            df[c] = df[c].map(lambda v: float(f"{v:.{fixed}g}") if pd.notna(v) else v)
    return pd.DataFrame(report)

In [ ]:
# 📚 Model catalogue as enforced by the Lexsi backend (xai/core/config.py, xai/utils/checks.py)
CATALOGUE = pd.DataFrame([
    # model,          classification, regression, support_set, shap_regression, commercial,  note
    ("OrionMoE",      True,  False, False, False, "yes",        "Lexsi mixture-of-experts, inference only"),
    ("OrionMSP",      True,  False, True,  False, "yes",        "Lexsi multi-scale sparse attention"),
    ("OrionBix",      True,  False, True,  False, "yes",        "Lexsi bi-axial attention"),
    ("OrionMSPv1.5",  True,  False, False, False, "yes",        "Lexsi"),
    ("TabICL",        True,  False, True,  False, "yes",        ""),
    ("TabICLv2",      True,  True,  False, True,  "yes",        ""),
    ("TabPFN",        True,  True,  False, True,  "unverified", ""),
    ("TabPFNv26",     True,  True,  True,  True,  "unverified", ""),
    ("TabPFNv3",      True,  True,  True,  True,  "no",         "needs a tabpfn_token"),
    ("TabDPT",        True,  True,  False, True,  "unverified", ""),
    ("Mitra",         True,  True,  False, False, "yes",        "no predict_proba"),
    ("ContextTab",    True,  True,  False, False, "no",         ""),
    ("Limix",         True,  True,  True,  True,  "no",         ""),
    ("TabFM",         True,  True,  False, False, "no",         "Google"),
    ("EXAONETabular", True,  False, False, False, "no",         "LG AI Research"),
    ("ILTM",          True,  True,  False, False, "yes",        ""),
    ("XRFM",          True,  True,  False, True,  "yes",        "trains from scratch"),
], columns=["model", "classification", "regression", "support_set", "shap_regression", "commercial", "note"]).set_index("model")


def xai_for(model, task):
    """Explainability methods the backend accepts for a simple build of `model`."""
    cap = CATALOGUE.loc[model]
    if task == "regression":
        return ["shap", "lime"] if cap.shap_regression else ["lime"]
    methods = ["shap", "lime"]
    if cap.support_set and RUN_SUPPORT_SET:
        methods.append("support_set")
    return methods

In [ ]:
# 🗂️ Organization → workspace → regression project
organization = lexsi.organization(ORGANIZATION)
workspace = organization.workspace(WORKSPACE)
project = get_or_create(workspace, PROJECT, "regression")
if TABPFN_TOKEN and "TabPFNv3" not in ROSTER:
    ROSTER.append("TabPFNv3")
assert all(CATALOGUE.loc[m, "regression"] for m in ROSTER), "every roster model must support regression"
display(CATALOGUE.loc[ROSTER].style.set_caption("Models in this demo (from the backend catalogue)"))

## 2 · Home Credit data

In [ ]:
# 📥 Load the processed Home Credit parquet files from Drive
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

UID = "SK_ID_CURR"
X_train = pd.read_parquet(os.path.join(DATA_DIR, "X_train.parquet"))
X_val   = pd.read_parquet(os.path.join(DATA_DIR, "X_val.parquet"))
# The default label (TARGET) is an outcome observed after the loan: never a feature for loan sizing.
for _df in (X_train, X_val):
    _df.drop(columns=["TARGET"], inplace=True, errors="ignore")
    _df[TARGET] = pd.to_numeric(_df[TARGET], errors="coerce")
X_train, X_val = X_train[X_train[TARGET].notna()], X_val[X_val[TARGET].notna()]

bad = [c for c in X_train.columns if re.search(r'[<>\[\]",]', str(c)) or str(c) != str(c).strip()]
X_train.drop(columns=bad, inplace=True, errors="ignore"); X_val.drop(columns=bad, inplace=True, errors="ignore")
print(f"train {X_train.shape} · validation {X_val.shape}")
display(X_train[TARGET].describe().to_frame().T.style.format("{:,.0f}"))

In [ ]:
# 📊 Target and driver dashboard
smp = X_train.sample(min(20000, len(X_train)), random_state=0)
fig = make_subplots(rows=1, cols=3, subplot_titles=("Credit amount distribution", "Credit vs income (log scale)",
                                                     "Median credit by contract type"))
fig.add_trace(go.Histogram(x=smp[TARGET], nbinsx=60, marker_color=BLUE, showlegend=False), 1, 1)
if "AMT_INCOME_TOTAL" in smp:
    fig.add_trace(go.Scattergl(x=smp["AMT_INCOME_TOTAL"], y=smp[TARGET], mode="markers",
                               marker=dict(size=3, color=ORANGE, opacity=0.3), showlegend=False), 1, 2)
    fig.update_xaxes(type="log", row=1, col=2); fig.update_yaxes(type="log", row=1, col=2)
if "NAME_CONTRACT_TYPE" in smp:
    med = smp.groupby("NAME_CONTRACT_TYPE")[TARGET].median()
    fig.add_trace(go.Bar(x=med.index.astype(str), y=med.values, marker_color=TEAL, showlegend=False), 1, 3)
fig.update_layout(template="plotly_white", height=430, title="Home Credit · what drives the loan amount?")
fig.show()

## 3 · Leakage guard

Some fields restate the answer: `AMT_GOODS_PRICE` is the price of the goods the loan pays for, and `AMT_ANNUITY` is computed from the credit amount. Left in, they give a near-perfect R² that means nothing. We drop them, any engineered feature built from them, and anything correlated above 0.95 with the target.

In [ ]:
# 🛡️ Find and exclude fields that leak the target
numeric = X_train.select_dtypes("number").drop(columns=[UID, TARGET], errors="ignore")
corr = numeric.sample(min(50000, len(numeric)), random_state=0).corrwith(X_train[TARGET]).abs().sort_values(ascending=False)
explicit = [c for c in ("AMT_GOODS_PRICE", "AMT_ANNUITY") if c in X_train]
derived = [c for c in X_train.columns if str(c).startswith("FE_") and re.search("CREDIT|ANNUITY|GOODS", str(c))]
LEAKY = sorted(set(explicit + derived + corr[corr > 0.95].index.tolist()))

top = corr.head(15)[::-1]
fig = go.Figure(go.Bar(x=top.values, y=top.index, orientation="h",
                       marker_color=[RED if c in LEAKY else GREY for c in top.index]))
fig.add_vline(x=0.95, line_dash="dot", line_color=DARK)
fig.update_layout(template="plotly_white", height=520, title="Correlation with AMT_CREDIT · red = excluded as leakage",
                  xaxis_title="|correlation|")
fig.show()
FEATURE_EXCLUDE = LEAKY
FEATURES = [c for c in X_train.columns if c not in (UID, TARGET, *FEATURE_EXCLUDE)]
print(f"excluded {len(LEAKY)}: {LEAKY}")
print(f"{len(FEATURES)} features remain")

## Data preparation lab

What is in the data, what needs fixing, which features carry signal, and what automated feature engineering adds. Every step is a plot.

In [ ]:
# 🧪 Data preparation lab · setup
DF_TRAIN, DF_HOLD, TARGET_COL, TASK = X_train, X_val, TARGET, "regression"
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_score
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

PREP_N = 8000                                                   # rows used for the analysis plots
prep = DF_TRAIN.sample(min(PREP_N, len(DF_TRAIN)), random_state=7).reset_index(drop=True)
y_prep = prep[TARGET_COL].astype(float) if TASK == "regression" else prep[TARGET_COL].astype(int)
NUMERIC = [c for c in FEATURES if pd.api.types.is_numeric_dtype(prep[c])]
CATEGORICAL = [c for c in FEATURES if c not in NUMERIC]
print(f"{len(DF_TRAIN):,} training rows · {len(FEATURES)} candidate features "
      f"({len(NUMERIC)} numeric, {len(CATEGORICAL)} categorical) · analysis sample {len(prep):,} rows")

In [ ]:
# 🔎 1 · Data quality profile: missing values, cardinality, constant columns, the backend's drop rule
miss = prep[FEATURES].isna().mean().mul(100)
uniq = prep[FEATURES].nunique() / len(prep)
const = [c for c in FEATURES if prep[c].nunique(dropna=True) <= 1]
binary = [c for c in NUMERIC if prep[c].dropna().isin([0, 1]).all()]
would_drop = uniq[uniq > 0.95].index.tolist()                   # platform rule: >95% unique values is treated as an identifier

kinds = pd.Series({"continuous": len(NUMERIC) - len(binary), "binary / one-hot": len(binary), "categorical": len(CATEGORICAL)})
top_miss = miss.sort_values(ascending=False).head(20)[::-1]
fig = make_subplots(rows=1, cols=3, column_widths=[0.22, 0.4, 0.38],
                    subplot_titles=("Feature types", "Most incomplete features (% missing)", "Cardinality: unique values ÷ rows"))
fig.add_trace(go.Bar(x=kinds.index, y=kinds.values, marker_color=[BLUE, TEAL, ORANGE], text=kinds.values, textposition="outside", showlegend=False), 1, 1)
fig.add_trace(go.Bar(x=top_miss.values, y=top_miss.index, orientation="h", marker_color=[RED if v > 50 else GREY for v in top_miss.values], showlegend=False), 1, 2)
fig.add_trace(go.Histogram(x=uniq.values, nbinsx=40, marker_color=BLUE, showlegend=False), 1, 3)
fig.add_vline(x=0.95, line_dash="dot", line_color=RED, row=1, col=3)
fig.update_layout(template="plotly_white", height=520, title="Data quality profile · red line: columns right of it are dropped by the platform as identifiers")
fig.show()
display(pd.DataFrame({"check": ["features with any missing value", "features more than 50% missing", "constant features",
                                "features above the 95%-unique rule", "duplicate rows in the sample"],
                      "count": [int((miss > 0).sum()), int((miss > 50).sum()), len(const), len(would_drop),
                                int(prep[FEATURES].duplicated().sum())]}).style.hide(axis="index"))

In [ ]:
# 📈 2 · Target and feature distributions: skew, outliers, and what a transform does
cont = [c for c in NUMERIC if c not in binary and prep[c].nunique() > 20]
skew = prep[cont].skew().abs().sort_values(ascending=False)
q1, q3 = prep[cont].quantile(0.25), prep[cont].quantile(0.75)
outlier = (((prep[cont] < q1 - 3 * (q3 - q1)) | (prep[cont] > q3 + 3 * (q3 - q1))).mean() * 100).sort_values(ascending=False)
signed_log = lambda s: np.sign(s) * np.log1p(np.abs(s))
show = skew.head(3).index.tolist()

fig = make_subplots(rows=2, cols=4, vertical_spacing=0.16,
                    subplot_titles=["Target"] + [f"{c[:26]} · raw" for c in show] + ["Most skewed features (|skew|)"] + [f"{c[:26]} · signed log" for c in show])
if TASK == "classification":
    vc = y_prep.value_counts().sort_index()
    fig.add_trace(go.Bar(x=[CLASS_NAMES[i] for i in vc.index], y=vc.values, marker_color=[BLUE, RED], text=vc.values, textposition="outside", showlegend=False), 1, 1)
else:
    fig.add_trace(go.Histogram(x=y_prep, nbinsx=60, marker_color=BLUE, showlegend=False), 1, 1)
for j, c in enumerate(show, 2):
    fig.add_trace(go.Histogram(x=prep[c], nbinsx=60, marker_color=GREY, showlegend=False), 1, j)
    fig.add_trace(go.Histogram(x=signed_log(prep[c]), nbinsx=60, marker_color=TEAL, showlegend=False), 2, j)
s10 = skew.head(10)[::-1]
fig.add_trace(go.Bar(x=s10.values, y=[c[:28] for c in s10.index], orientation="h", marker_color=ORANGE, showlegend=False), 2, 1)
fig.update_layout(template="plotly_white", height=640, title="Distributions · heavy tails before and after a signed-log transform")
fig.show()
print(f"{int((skew > 2).sum())} of {len(cont)} continuous features are heavily skewed (|skew| > 2) · "
      f"{int((outlier > 1).sum())} have more than 1% extreme outliers")
print("Tabular foundation models apply their own robust scaling, so no manual transform is needed before upload; "
      "the plot shows why that scaling matters.")

In [ ]:
# 🎯 3 · Which features carry signal, and which are redundant
Xn = prep[NUMERIC].replace([np.inf, -np.inf], np.nan)
Xn = Xn.fillna(Xn.median()).fillna(0)
mi_fn = mutual_info_regression if TASK == "regression" else mutual_info_classif
MI = pd.Series(mi_fn(Xn, y_prep, random_state=0), index=NUMERIC).sort_values(ascending=False)
top = MI.head(20).index.tolist()
corr = Xn[top].corr()
full_corr = Xn[MI.head(60).index].corr().abs()
pairs = (full_corr.where(np.triu(np.ones(full_corr.shape, bool), 1)).stack().sort_values(ascending=False))
redundant = pairs[pairs > 0.9]

fig = make_subplots(rows=1, cols=2, column_widths=[0.4, 0.6], subplot_titles=("Mutual information with the target (top 20)", "Correlation between those features"))
fig.add_trace(go.Bar(x=MI.head(20).values[::-1], y=[c[:30] for c in top[::-1]], orientation="h", marker_color=BLUE, showlegend=False), 1, 1)
fig.add_trace(go.Heatmap(z=corr.values, x=[c[:18] for c in top], y=[c[:18] for c in top], colorscale="RdBu_r", zmin=-1, zmax=1, showscale=True), 1, 2)
fig.update_layout(template="plotly_white", height=640, title="Signal and redundancy")
fig.show()
print(f"{int((MI < 0.001).sum())} of {len(NUMERIC)} numeric features carry almost no signal · {len(redundant)} highly correlated pairs (|r| > 0.9) among the top 60")
if len(redundant):
    display(redundant.head(8).rename("abs correlation").reset_index().rename(columns={"level_0": "feature A", "level_1": "feature B"}))

# How the target moves across the range of the four strongest features
fig = make_subplots(rows=1, cols=4, subplot_titles=[c[:28] for c in top[:4]])
for j, c in enumerate(top[:4], 1):
    b = pd.qcut(prep[c].rank(method="first"), 10, labels=False)
    g = pd.DataFrame({"b": b, "x": prep[c], "y": y_prep}).groupby("b").agg(x=("x", "median"), y=("y", "mean"))
    fig.add_trace(go.Scatter(x=g.x, y=g.y * (100 if TASK == "classification" else 1), mode="lines+markers", line=dict(color=ORANGE, width=3), showlegend=False), 1, j)
fig.update_yaxes(title_text="default rate (%)" if TASK == "classification" else f"mean {TARGET_COL}", row=1, col=1)
fig.update_layout(template="plotly_white", height=380, title="Target response across deciles of the strongest features")
fig.show()

In [ ]:
# 🛠️ 4 · Automated feature engineering: generate candidates, keep the ones that add signal, measure the gain
TOP_K, KEEP = 8, 10
base = [c for c in MI.index if c in cont][:TOP_K]
recipes = []                                                      # (name, operation, a, b)
for i, a in enumerate(base):
    for b in base[i + 1:]:
        recipes += [(f"fe_prod__{a}__{b}", "prod", a, b), (f"fe_diff__{a}__{b}", "diff", a, b)]
        if (prep[b] > 0).all():
            recipes.append((f"fe_ratio__{a}__{b}", "ratio", a, b))
recipes.append(("fe_missing_count", "missing", None, None))


def apply_recipes(df, recs):
    out = {}
    for name, op, a, b in recs:
        if op == "prod":
            out[name] = df[a] * df[b]
        elif op == "diff":
            out[name] = df[a] - df[b]
        elif op == "ratio":
            out[name] = df[a] / df[b].where(df[b] > 0)
        elif op == "missing":
            out[name] = df[[c for c in FEATURES if c in df.columns]].isna().sum(axis=1)
    return pd.DataFrame(out, index=df.index).replace([np.inf, -np.inf], np.nan)


cand = apply_recipes(prep, recipes)
cand_f = cand.fillna(cand.median()).fillna(0)
cand_mi = pd.Series(mi_fn(cand_f, y_prep, random_state=0), index=cand.columns)
parent_mi = pd.Series({n: max(MI.get(a, 0), MI.get(b, 0)) if a else 0 for n, _, a, b in recipes})
gain = (cand_mi - parent_mi).sort_values(ascending=False)
KEPT = gain.head(KEEP).index.tolist()
KEPT_RECIPES = [r for r in recipes if r[0] in KEPT]

# Probe model: does the engineered set improve a quick cross-validated model?
Model = HistGradientBoostingRegressor if TASK == "regression" else HistGradientBoostingClassifier
scoring = "r2" if TASK == "regression" else "roc_auc"
cv = lambda X: cross_val_score(Model(max_iter=120, random_state=0), X, y_prep, cv=3, scoring=scoring).mean()
score_raw = cv(Xn)
score_fe = cv(pd.concat([Xn, cand_f[KEPT]], axis=1))

fig = make_subplots(rows=1, cols=3, column_widths=[0.36, 0.4, 0.24],
                    subplot_titles=(f"{len(recipes)} candidates: signal vs best parent", "Kept features: information gain over parents", f"Probe model ({scoring}, 3-fold)"))
fig.add_trace(go.Scatter(x=parent_mi, y=cand_mi, mode="markers", showlegend=False,
                         marker=dict(color=[ORANGE if n in KEPT else GREY for n in cand_mi.index], size=[10 if n in KEPT else 5 for n in cand_mi.index])), 1, 1)
mx = float(max(cand_mi.max(), parent_mi.max()))
fig.add_trace(go.Scatter(x=[0, mx], y=[0, mx], line=dict(color=DARK, dash="dot"), showlegend=False), 1, 1)
g = gain[KEPT][::-1]
fig.add_trace(go.Bar(x=g.values, y=[n.replace("fe_", "").replace("__", " · ")[:44] for n in g.index], orientation="h", marker_color=ORANGE, showlegend=False), 1, 2)
fig.add_trace(go.Bar(x=["raw features", "+ engineered"], y=[score_raw, score_fe], marker_color=[GREY, ORANGE],
                     text=[f"{score_raw:.4f}", f"{score_fe:.4f}"], textposition="outside", showlegend=False), 1, 3)
fig.update_xaxes(title_text="best parent's mutual information", row=1, col=1); fig.update_yaxes(title_text="candidate's mutual information", row=1, col=1)
fig.update_yaxes(range=[min(score_raw, score_fe) * 0.97, max(score_raw, score_fe) * 1.02], row=1, col=3)
fig.update_layout(template="plotly_white", height=560, title="Automated feature engineering · orange = kept")
fig.show()
print(f"kept {len(KEPT)} engineered features · probe {scoring}: {score_raw:.4f} → {score_fe:.4f} ({score_fe - score_raw:+.4f})")

In [ ]:
# ✅ 5 · Apply the recipe to the full tables, then summarise the preparation pipeline
ADD_ENGINEERED = score_fe > score_raw          # only ship engineered features if the probe says they help; set True/False to override
print("engineered features", "ADDED: the probe improved" if ADD_ENGINEERED else "NOT added: the probe did not improve, raw features go to the platform")
if ADD_ENGINEERED:
    for _df in (DF_TRAIN, DF_HOLD):
        fe = apply_recipes(_df, KEPT_RECIPES)
        for c in fe.columns:
            _df[c] = fe[c].round(6)
    FEATURES = [c for c in DF_TRAIN.columns if c not in (UID, TARGET_COL, *FEATURE_EXCLUDE)]

n_raw = DF_TRAIN.shape[1] - (len(KEPT) if ADD_ENGINEERED else 0)
stages = [("columns in the source table", n_raw), ("− identifier and target", n_raw - 2),
          ("− excluded (identifiers, leakage)", n_raw - 2 - len(FEATURE_EXCLUDE)),
          ("− constant", n_raw - 2 - len(FEATURE_EXCLUDE) - len(const)),
          ("+ engineered", n_raw - 2 - len(FEATURE_EXCLUDE) - len(const) + (len(KEPT) if ADD_ENGINEERED else 0))]

# 2-D map of the data: do the classes (or target levels) separate?
Z = PCA(n_components=2, random_state=0).fit_transform(StandardScaler().fit_transform(Xn[MI.head(30).index]))
fig = make_subplots(rows=1, cols=2, column_widths=[0.42, 0.58], subplot_titles=("Feature pipeline", "The data in two dimensions (PCA of the 30 strongest features)"))
fig.add_trace(go.Funnel(y=[s for s, _ in stages], x=[v for _, v in stages], marker_color=[GREY, GREY, RED, RED, ORANGE], textinfo="value", showlegend=False), 1, 1)
if TASK == "classification":
    for cls, col in ((0, BLUE), (1, RED)):
        m = (y_prep == cls).to_numpy()
        fig.add_trace(go.Scattergl(x=Z[m, 0], y=Z[m, 1], mode="markers", name=CLASS_NAMES[cls], marker=dict(size=4, color=col, opacity=0.45)), 1, 2)
else:
    fig.add_trace(go.Scattergl(x=Z[:, 0], y=Z[:, 1], mode="markers", showlegend=False,
                               marker=dict(size=4, color=np.log1p(y_prep.clip(lower=0)), colorscale="Viridis", opacity=0.6, colorbar=dict(title="log target"))), 1, 2)
fig.update_layout(template="plotly_white", height=540, title="From source table to model-ready features", legend=dict(orientation="h", y=-0.1))
fig.show()
print(f"{len(FEATURES)} features go to the platform ({len(KEPT) if ADD_ENGINEERED else 0} engineered). "
      "The platform then encodes categoricals, fills missing values and applies each model's own preprocessing.")

In [ ]:
# ✂️ Sample, backend pre-flight check, CSVs
for _df in (X_train, X_val):                       # spaces in column names break the backend with lexsi-sdk 0.1.69
    _df.columns = _df.columns.str.replace(" ", "_")
FEATURE_EXCLUDE = [c.replace(" ", "_") for c in FEATURE_EXCLUDE]
FEATURES = [c for c in X_train.columns if c not in (UID, TARGET, *FEATURE_EXCLUDE)]
train_sample = X_train.sample(min(SAMPLE_TRAIN_N, len(X_train)), random_state=42).reset_index(drop=True)
test_sample  = X_val.sample(min(SAMPLE_TEST_N, len(X_val)), random_state=42).reset_index(drop=True)
both = pd.concat([train_sample, test_sample])
fixes = preflight_high_cardinality(both, FEATURES)
train_sample, test_sample = both.iloc[:len(train_sample)].copy(), both.iloc[len(train_sample):].copy()
display(fixes if len(fixes) else Markdown("No feature would be dropped by the backend's 95%-unique rule."))
assert train_sample[TARGET].nunique() >= 100, "the backend needs at least 100 distinct target values"

TRAIN_CSV, TEST_CSV = "/content/hc_reg_demo_train.csv", "/content/hc_reg_demo_test.csv"
train_sample.to_csv(TRAIN_CSV, index=False)
test_sample.to_csv(TEST_CSV, index=False)
print(f"train sample {train_sample.shape} · test sample {test_sample.shape}")

In [ ]:
# ⬆️ Upload once
config = {"unique_identifier": UID, "true_label": TARGET, "pred_label": "",
          "feature_exclude": FEATURE_EXCLUDE, "drop_duplicate_uid": True, "handle_errors": True}
upload_once(project, TRAIN_CSV, "Training", config=config, compute_type=UPLOAD_COMPUTE)
upload_once(project, TEST_CSV, "Test")
display(project.files()[["file_name", "tag", "status", "created_at"]])
print("tags:", project.tags())

## 4 · Zero-shot training

Every model runs with `tunning_strategy="inference"`, requests the explanations the backend accepts for regression, and is scored by the platform on the **Test** tag. The leakage exclusions are passed on every build.

In [ ]:
# 🚀 One simple build per model
BUILDS = {}
for model in ROSTER:
    name = train(project, f"{model} · zero-shot", model=model, training_strategy="simple",
                 compute_type=COMPUTE, tunning_strategy="inference",
                 data_config=base_data_config(xai_for(model, "regression")), model_config=model_config_for(model))
    if name:
        BUILDS[model] = name
print("\nbuilt:", BUILDS)

## 5 · Model comparison

In [ ]:
# 🏁 Platform metrics side by side
SUMMARIES = {m: summary_of(project, n) for m, n in BUILDS.items()}
rows = []
for m, s in SUMMARIES.items():
    em = eval_metrics(s)
    rows.append({"model": m, "build": BUILDS[m], "R2": em.get("r2_score"), "RMSE": em.get("RootMeanSquaredError"),
                 "MAE": em.get("Mean_Absolute_Error"), "MAPE": em.get("Mean_Absolute_Per_Error"),
                 "features used": len(features_used(s)), "commercial": CATALOGUE.loc[m, "commercial"]})
board = pd.DataFrame(rows).set_index("model").sort_values("R2", ascending=False)
display(board.style.format({"R2": "{:.3f}", "RMSE": "{:,.0f}", "MAE": "{:,.0f}", "MAPE": "{:.3f}"}, na_rep="-")
        .background_gradient(subset=["R2"], cmap="Blues").set_caption("Scored by the platform on the Test tag"))
fig = make_subplots(rows=1, cols=2, subplot_titles=("R² (higher is better)", "MAE (lower is better)"))
fig.add_trace(go.Bar(x=board.index, y=board["R2"], marker_color=BLUE, text=board["R2"], texttemplate="%{text:.3f}"), 1, 1)
fig.add_trace(go.Bar(x=board.index, y=board["MAE"], marker_color=ORANGE, text=board["MAE"], texttemplate="%{text:,.0f}"), 1, 2)
fig.update_layout(template="plotly_white", height=430, showlegend=False,
                  title="Zero-shot foundation models · held-out applicants")
fig.show()
BEST = board["R2"].astype(float).fillna(-1e9).idxmax()
display(Markdown(f"**Best by R²: `{BEST}` ({BUILDS[BEST]})**"))
diag = modelling_info(SUMMARIES[BEST]).get("regression_diagnostics")
if diag:
    display(Markdown("**Regression diagnostics stored by the platform** (intervals, interval calibration, residuals)"))
    display(pd.json_normalize(diag).T.head(30))

## AutoML race

Classic models with automatic hyperparameter search against zero-shot foundation models, on one leaderboard.

In [ ]:
# 🤖 AutoML race: classic ML with automatic hyperparameter search vs zero-shot foundation models
RUN_AUTOML = True
CLASSIC = ["XGBoost", "LGBoost", "CatBoost", "RandomForest"]
CLASSIC_COMPUTE = UPLOAD_COMPUTE                                   # classic models train on CPU

def metric_lookup(summary, *names):
    """First matching metric from a model summary, whatever the key spelling."""
    em = {re.sub(r"[^a-z0-9]", "", k.lower()): v for k, v in eval_metrics(summary).items() if isinstance(v, (int, float))}
    for n in names:
        k = re.sub(r"[^a-z0-9]", "", n.lower())
        if k in em:
            return em[k]
    return None

HEADLINE = ("r2_score", "r2") if TASK == "regression" else ("RocAucScore", "roc_auc", "auc")
SECOND = ("RootMeanSquaredError", "rmse") if TASK == "regression" else ("F1Score", "f1")
CLASSIC_BUILDS = {}
if RUN_AUTOML:
    for model in CLASSIC:
        name = train(project, f"{model} · AutoML (Optuna)", model=model, training_strategy="simple", compute_type=CLASSIC_COMPUTE,
                     data_config={"tags": ["Training"], "test_tags": ["Test"], "feature_exclude": FEATURE_EXCLUDE,
                                  "use_optuna": True, "xai_method": ["shap"]})
        if name:
            CLASSIC_BUILDS[model] = name

race = []
for family, builds in (("Foundation model (zero-shot)", BUILDS), ("Classic ML (AutoML)", CLASSIC_BUILDS)):
    for model, name in builds.items():
        s = summary_of(project, name)
        mins = next((r["minutes"] for r in RUNS if r["created"] == name), None)
        race.append({"model": model, "family": family, "build": name, "headline": metric_lookup(s, *HEADLINE),
                     "second": metric_lookup(s, *SECOND), "minutes": mins})
race = pd.DataFrame(race).sort_values("headline", ascending=False).reset_index(drop=True)
display(race.rename(columns={"headline": HEADLINE[0], "second": SECOND[0]}).style.format(precision=3, na_rep="-")
        .set_caption("One leaderboard: every model scored by the platform on the same Test tag"))

FAM = {"Foundation model (zero-shot)": BLUE, "Classic ML (AutoML)": GREY}
fig = make_subplots(rows=1, cols=2, subplot_titles=(f"{HEADLINE[0]} on the Test tag", "Accuracy vs build time (minutes)"))
r = race.sort_values("headline")
fig.add_trace(go.Bar(x=r.headline, y=r.model, orientation="h", marker_color=[FAM[f] for f in r.family], text=r.headline, texttemplate="%{text:.3f}", showlegend=False), 1, 1)
for fam, g in race.groupby("family"):
    fig.add_trace(go.Scatter(x=g.minutes, y=g.headline, mode="markers+text", text=g.model, textposition="top center", name=fam,
                             marker=dict(size=14, color=FAM[fam])), 1, 2)
fig.update_layout(template="plotly_white", height=480, title="AutoML leaderboard · foundation models need no tuning run", legend=dict(orientation="h", y=-0.15))
fig.show()

## 6 · Explainability across models

In [ ]:
# 🔍 Global SHAP per model, cross-model heatmap and SHAP vs LIME
shap = {m: gfi(s, "shap") for m, s in SUMMARIES.items() if gfi(s, "shap")}
lime = {m: gfi(s, "lime") for m, s in SUMMARIES.items() if gfi(s, "lime")}
if shap:
    top_bar(shap.get(BEST) or next(iter(shap.values())), f"Global SHAP · {BEST}").show()
    mat = pd.DataFrame(shap).fillna(0)
    mat = mat.div(mat.max())
    top = mat.mean(axis=1).sort_values(ascending=False).head(15).index
    fig = px.imshow(mat.loc[top], color_continuous_scale="Blues", aspect="auto", text_auto=".2f",
                    title="Relative SHAP importance · top 15 features across models")
    fig.update_layout(height=620, template="plotly_white"); fig.show()
if BEST in shap and BEST in lime:
    b = pd.DataFrame({"SHAP": pd.Series(shap[BEST]), "LIME": pd.Series(lime[BEST])}).fillna(0)
    b = b.div(b.max())
    t = b.max(axis=1).sort_values(ascending=False).head(12).index[::-1]
    fig = go.Figure([go.Bar(y=t, x=b.loc[t, "SHAP"], name="SHAP", orientation="h", marker_color=BLUE),
                     go.Bar(y=t, x=b.loc[t, "LIME"], name="LIME", orientation="h", marker_color=ORANGE)])
    fig.update_layout(template="plotly_white", barmode="group", height=560, title=f"SHAP vs LIME · {BEST}")
    fig.show()

## 7 · Batch scoring on held-out applicants

In [ ]:
# 📦 Score the Test tag with the best model, evaluate locally
ensure_inference(project, BUILDS[BEST])
scored = project.model_inference(tag="Test", model_name=BUILDS[BEST], compute_type=COMPUTE)
scored = scored.merge(test_sample[[UID, TARGET]].rename(columns={TARGET: "actual"}), on=UID, how="left")
pred, actual = scored["Predicted_value_AutoML"].astype(float), scored["actual"].astype(float)
local = regression_dashboard(actual, pred, f"{BUILDS[BEST]} · {len(scored):,} held-out applicants", unit="(currency)")
display(pd.Series(local, name="local evaluation").to_frame().style.format("{:,.3f}"))

In [ ]:
# 💼 Business view: how often is the recommended amount close to the approved amount?
ape = (pred - actual).abs() / actual.clip(lower=1) * 100
bands = pd.cut(ape, [0, 10, 20, 35, 50, np.inf], labels=["within 10%", "10–20%", "20–35%", "35–50%", "over 50%"])
share = bands.value_counts(normalize=True).reindex(bands.cat.categories).mul(100)
fig = go.Figure(go.Bar(x=share.index.astype(str), y=share.values, marker_color=[TEAL, BLUE, GREY, ORANGE, RED],
                       text=share.values, texttemplate="%{text:.0f}%"))
fig.update_layout(template="plotly_white", height=420, yaxis_title="% of applicants",
                  title=f"{share.iloc[:2].sum():.0f}% of recommendations land within 20% of the approved amount")
fig.show()

## 8 · One applicant, explained

In [ ]:
# 🧑‍💼 Explain the recommendation for a held-out applicant with a large approved amount
CASE_UID = str(scored.sort_values("actual", ascending=False)[UID].iloc[len(scored) // 20])
case = project.case_predict(unique_identifier=CASE_UID, tag="Test", model_name=BUILDS[BEST], xai=["shap", "lime"])
display(case.inference_output())
sh, li = case_attributions(case, "shap"), case_attributions(case, "lime")
if sh:
    signed_bar(sh, f"SHAP · applicant {CASE_UID} · red raises the recommended amount").show()
if li:
    signed_bar(li, f"LIME · applicant {CASE_UID}").show()
top = pd.Series(sh or li).abs().sort_values(ascending=False).head(6).index
row = test_sample.assign(_k=test_sample[UID].astype(str)).set_index("_k").loc[CASE_UID]
display(pd.DataFrame({"applicant": row[top], "portfolio median": train_sample[top].median(numeric_only=True)})
        .style.set_caption("The applicant's top drivers next to the portfolio median"))

## 9 · Advanced builds

* **Ensemble** — greedy selection over the zero-shot models, optimising RMSE (explained with LIME).
* **Distillation** — the best model's knowledge compressed into a fast LightGBM student for real-time scoring.
* **Leaderboard** — the platform races the models on an internal hold-out by R² and refits the winner.

In [ ]:
# 🤝 Ensemble · 🧪 Distillation · 🏆 Leaderboard
ADV = {}
pool = list(BUILDS)[:3]
if RUN_ENSEMBLE and len(pool) >= 2:
    ADV["Ensemble"] = train(project, "Ensemble · greedy RMSE", model=pool, training_strategy="ensemble",
                            compute_type=COMPUTE, tunning_strategy="inference",
                            data_config=base_data_config(["lime"]), model_config=model_config_for(pool),
                            ensemble_config={"ensemble_strategy": "greedy_selection", "metric": "rmse",
                                             "weight_scheme": "performance", "cv_folds": 5,
                                             "holdout_fraction": 0.2, "greedy_ensemble_size": 50})
if RUN_DISTILLATION:
    ADV["Distilled"] = train(project, f"Distillation · {BEST} → LightGBM", model=BEST, training_strategy="distillation",
                             compute_type=COMPUTE, data_config=base_data_config(["shap", "lime"]),
                             model_config=model_config_for(BEST),
                             distillation_config={"student": "lgbm", "gbdt_backend": "lgbm", "temperature": 3.0,
                                                  "alpha": 0.7, "n_folds": 5, "adaptive_temperature": True,
                                                  "confidence_weighting": True})
if RUN_LEADERBOARD and len(BUILDS) >= 2:
    ADV["Leaderboard"] = train(project, "Leaderboard · R²", model=list(BUILDS), training_strategy="leaderboard",
                               compute_type=COMPUTE, tunning_strategy="inference",
                               data_config=base_data_config(["lime"]), model_config=model_config_for(list(BUILDS)),
                               leaderboard_config={"rank_by": "r2_score"})
ADV = {k: v for k, v in ADV.items() if v}
print(ADV)

In [ ]:
# 📊 Everything built today, on the same Test tag
rows = []
for label, name in {**BUILDS, **ADV}.items():
    s = summary_of(project, name)
    em = eval_metrics(s)
    rows.append({"build": label, "model": name, "R2": em.get("r2_score"), "RMSE": em.get("RootMeanSquaredError"),
                 "MAE": em.get("Mean_Absolute_Error")})
    mi = modelling_info(s)
    if label == "Leaderboard" and mi.get("leaderboard"):
        display(Markdown(f"**Leaderboard winner: `{mi.get('winner_model_name')}`**"))
        display(pd.DataFrame(mi["leaderboard"]))
final = pd.DataFrame(rows).set_index("build")
display(final.style.format({"R2": "{:.3f}", "RMSE": "{:,.0f}", "MAE": "{:,.0f}"}, na_rep="-")
        .background_gradient(subset=["R2"], cmap="Blues"))
fig = px.bar(final.reset_index(), x="build", y="R2", color="build", text_auto=".3f",
             title="R² on held-out applicants · zero-shot models vs advanced builds")
fig.update_layout(template="plotly_white", height=430, showlegend=False); fig.show()

## Modelling at scale and the retraining agent

How accuracy, build time and scoring speed move with data size, then a small agent that watches new data, decides whether to retrain, and promotes a challenger only if it wins.

In [ ]:
# 🏗️ Modelling at scale: learning curve, build time, scoring throughput, and the distilled student
RUN_SCALE = True
FRACTIONS = [0.25, 0.5]                                           # the full build (1.0) is the one trained above
curve = [{"fraction": 1.0, "build": BUILDS[BEST]}]
if RUN_SCALE:
    for frac in FRACTIONS:
        name = train(project, f"{BEST} · {int(frac * 100)}% of rows", model=BEST, training_strategy="simple", compute_type=COMPUTE,
                     tunning_strategy="inference", model_config=model_config_for(BEST),
                     data_config={**base_data_config(["lime"]), "sample_percentage": frac})
        if name:
            curve.append({"fraction": frac, "build": name})
for c in curve:
    s = summary_of(project, c["build"])
    c["metric"] = metric_lookup(s, *HEADLINE)
    c["rows"] = (modelling_info(s).get("train_shape") or [None])[0]
    c["minutes"] = next((r["minutes"] for r in RUNS if r["created"] == c["build"]), None)
curve = pd.DataFrame(curve).sort_values("fraction")

# Scoring throughput: the foundation model against its distilled student (if one was built)
speed = []
candidates = {BEST: BUILDS[BEST], **{k: v for k, v in ADV.items() if "istil" in k or "PAD" in k}}
for label, name in candidates.items():
    try:
        ensure_inference(project, name)
        t0 = time.time()
        out = project.model_inference(tag="Test", model_name=name, compute_type=COMPUTE)
        speed.append({"model": label, "rows": len(out), "seconds": time.time() - t0})
    except Exception as exc:
        print(f"throughput skipped for {label}: {str(exc)[:100]}")
speed = pd.DataFrame(speed)
if len(speed):
    speed["rows_per_second"] = speed.rows / speed.seconds
ensure_inference(project, BUILDS[BEST])

fig = make_subplots(rows=1, cols=3, subplot_titles=(f"Learning curve · {HEADLINE[0]} vs context rows", "Build time (minutes)", "Batch scoring throughput (rows/second, end to end)"))
x = curve.rows.fillna(curve.fraction).astype(float)
fig.add_trace(go.Scatter(x=x, y=curve.metric, mode="lines+markers", line=dict(color=BLUE, width=3), showlegend=False), 1, 1)
fig.add_trace(go.Bar(x=[f"{int(f * 100)}%" for f in curve.fraction], y=curve.minutes, marker_color=GREY, showlegend=False), 1, 2)
if len(speed):
    fig.add_trace(go.Bar(x=speed.model, y=speed.rows_per_second, marker_color=[BLUE] + [ORANGE] * (len(speed) - 1),
                         text=speed.rows_per_second, texttemplate="%{text:.0f}", showlegend=False), 1, 3)
fig.update_layout(template="plotly_white", height=440, title=f"Scale profile · {BEST}")
fig.show()
display(pd.DataFrame({"lever": ["Context size", "Row sampling", "Compute", "Ensembles", "Distillation", "Batch scoring"],
                      "how it is used here": ["foundation models read the training rows as context; the platform caps a build at 10,000 rows",
                                              "sample_percentage in data_config sets the share of rows per build",
                                              f"GPU queue {COMPUTE} for foundation models, CPU {UPLOAD_COMPUTE} for classic models and uploads",
                                              "several foundation models combined by greedy selection",
                                              "a tree or MLP student learns the foundation model's predictions for cheap, fast scoring",
                                              "model_inference scores a whole tag in one call"]}).style.hide(axis="index"))

In [ ]:
# 🔁 Retraining agent: observe → decide → act → verify → promote
# A small policy agent. It watches a new batch for drift and decay, decides whether to retrain,
# trains a challenger on the platform, checks it on fresh data and promotes it only if it wins.
RUN_AGENT = True
PSI_ALERT, DECAY_ALERT, PROMOTE_MARGIN = 0.20, 0.02, 0.003


def psi(ref, new, bins=10):
    """Population stability index of one feature: 0 = identical, above 0.2 = material drift."""
    edges = np.unique(np.quantile(ref.dropna(), np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return 0.0
    edges[0], edges[-1] = -np.inf, np.inf
    a = np.histogram(ref.dropna(), edges)[0] / max(ref.notna().sum(), 1) + 1e-4
    b = np.histogram(new.dropna(), edges)[0] / max(new.notna().sum(), 1) + 1e-4
    return float(np.sum((b - a) * np.log(b / a)))


def score_frame(scored_df, truth):
    """Headline metric from a platform inference frame (probability or prediction column) against the truth."""
    m = scored_df.merge(truth[[UID, TARGET_COL]].rename(columns={TARGET_COL: "_truth"}), on=UID)
    if TASK == "regression":
        return float(skm.r2_score(m["_truth"].astype(float), m["Predicted_value_AutoML"].astype(float)))
    return float(skm.roc_auc_score((m["_truth"].astype(str) == "1").astype(int), m["pred_proba_AutoML"].astype(float)))


class RetrainAgent:
    def __init__(self, project, champion_model, champion_build):
        self.project, self.model, self.champion, self.challenger = project, champion_model, champion_build, None
        self.trace, self.state = [], {}

    def log(self, step, finding, decision=""):
        self.trace.append({"step": step, "finding": finding, "decision": decision, "at": pd.Timestamp.now(), "seconds": 0.0})
        print(f"[{step}] {finding}" + (f" → {decision}" if decision else ""))

    def _timed(self, fn, *a, **k):
        t0 = time.time(); out = fn(*a, **k); self.trace[-1]["seconds"] = round(time.time() - t0, 1); return out

    def _upload(self, df, tag):
        path = f"/content/agent_{tag.lower()}.csv"
        df.to_csv(path, index=False)
        upload_once(self.project, path, tag)

    def _score(self, build, tag, truth):
        ensure_inference(self.project, build)
        return score_frame(self.project.model_inference(tag=tag, model_name=build, compute_type=COMPUTE), truth)

    def observe(self, reference, batch, batch_tag, baseline):
        num = [c for c in FEATURES if c in batch.columns and pd.api.types.is_numeric_dtype(batch[c]) and reference[c].nunique() > 10]
        self.state["psi"] = pd.Series({c: psi(reference[c], batch[c]) for c in num}).sort_values(ascending=False)
        drifted = int((self.state["psi"] > PSI_ALERT).sum())
        self.log("observe · drift", f"{drifted} of {len(num)} features above PSI {PSI_ALERT} (max {self.state['psi'].iloc[0]:.2f} on {self.state['psi'].index[0]})")
        self.log("observe · performance", "scoring the new batch with the champion")
        self._timed(self._upload, batch, batch_tag)
        live = self._score(self.champion, batch_tag, batch)
        self.state.update(baseline=baseline, live=live, drifted=drifted, batch_tag=batch_tag)
        self.trace[-1]["finding"] = f"champion {HEADLINE[0]} {baseline:.3f} on Test → {live:.3f} on the new batch ({live - baseline:+.3f})"
        print("   ", self.trace[-1]["finding"])
        return self

    def decide(self):
        s = self.state
        reasons = ([f"{s['drifted']} drifted features"] if s["drifted"] >= 1 else []) + \
                  ([f"performance down {s['baseline'] - s['live']:.3f}"] if s["baseline"] - s["live"] > DECAY_ALERT else [])
        s["retrain"] = bool(reasons)
        self.log("decide", "; ".join(reasons) or "no material drift or decay", "RETRAIN" if reasons else "KEEP CHAMPION")
        return self

    def act(self):
        if not self.state["retrain"]:
            return self
        self.log("act", f"training a challenger {self.model} on Training + {self.state['batch_tag']}")
        cfg = {**base_data_config(xai_for(self.model, TASK)), "tags": ["Training", self.state["batch_tag"]]}
        self.challenger = self._timed(train, self.project, f"{self.model} · challenger", model=self.model, training_strategy="simple",
                                      compute_type=COMPUTE, tunning_strategy="inference", data_config=cfg, model_config=model_config_for(self.model))
        self.trace[-1]["decision"] = self.challenger or "build failed"
        return self

    def verify(self, recent, recent_tag):
        if not self.challenger:
            return self
        self.log("verify", f"scoring champion and challenger on unseen recent data ({len(recent):,} rows)")
        self._upload(recent, recent_tag)
        t0 = time.time()
        self.state["champ_recent"] = self._score(self.champion, recent_tag, recent)
        self.state["chall_recent"] = self._score(self.challenger, recent_tag, recent)
        self.trace[-1]["seconds"] = round(time.time() - t0, 1)
        self.trace[-1]["finding"] = f"champion {self.state['champ_recent']:.3f} vs challenger {self.state['chall_recent']:.3f} on recent data"
        print("   ", self.trace[-1]["finding"])
        return self

    def promote(self):
        s = self.state
        win = bool(self.challenger) and s.get("chall_recent", -np.inf) - s.get("champ_recent", np.inf) >= PROMOTE_MARGIN
        if win:
            self.project.activate_model(self.challenger)
        else:
            self.project.activate_model(self.champion)
        s["promoted"] = win
        self.log("promote", f"margin required {PROMOTE_MARGIN}", f"PROMOTED {self.challenger}" if win else f"KEPT {self.champion}")
        return self


# --- New data arrives: unseen rows whose mix has shifted towards one end of the strongest feature -------------
used = set(test_sample[UID])
pool = DF_HOLD[~DF_HOLD[UID].isin(used)]
drift_feature = next(c for c in MI.index if c in pool.columns and pool[c].nunique() > 20)
z = (pool[drift_feature] - pool[drift_feature].mean()) / (pool[drift_feature].std() + 1e-9)
shifted = pool.sample(min(2400, len(pool)), weights=np.exp(1.2 * z.fillna(0).clip(-3, 3)), random_state=5)   # real rows, shifted mix
new_batch, recent = shifted.iloc[:1200].copy(), shifted.iloc[1200:].copy()
print(f"new batch: {len(new_batch):,} rows · recent hold-out: {len(recent):,} rows · mix shifted on '{drift_feature}'")

if RUN_AGENT:
    baseline = score_frame(scored, test_sample)
    agent = RetrainAgent(project, BEST, BUILDS[BEST])
    agent.observe(train_sample, new_batch, "Batch1", baseline).decide().act().verify(recent, "Recent1").promote()

In [ ]:
# 📊 Retraining agent · what it saw, what it decided, what changed
if RUN_AGENT:
    s, tr = agent.state, pd.DataFrame(agent.trace)
    top_psi = s["psi"].head(12)[::-1]
    fig = make_subplots(rows=2, cols=2, vertical_spacing=0.18, specs=[[{}, {}], [{}, {}]],
                        subplot_titles=("Drift by feature (PSI)", f"Most drifted feature · {s['psi'].index[0][:34]}",
                                        f"{HEADLINE[0]}: champion vs challenger", "Agent timeline (seconds per step)"))
    fig.add_trace(go.Bar(x=top_psi.values, y=[c[:30] for c in top_psi.index], orientation="h", showlegend=False,
                         marker_color=[RED if v > PSI_ALERT else ORANGE if v > 0.1 else GREY for v in top_psi.values]), 1, 1)
    fig.add_vline(x=PSI_ALERT, line_dash="dot", line_color=RED, row=1, col=1)
    f0 = s["psi"].index[0]
    fig.add_trace(go.Histogram(x=train_sample[f0], histnorm="probability", nbinsx=40, name="training data", marker_color=GREY, opacity=0.7), 1, 2)
    fig.add_trace(go.Histogram(x=new_batch[f0], histnorm="probability", nbinsx=40, name="new batch", marker_color=RED, opacity=0.6), 1, 2)
    bars = [("champion · Test", s["baseline"], GREY), ("champion · new batch", s["live"], RED)]
    if "chall_recent" in s:
        bars += [("champion · recent", s["champ_recent"], GREY), ("challenger · recent", s["chall_recent"], BLUE)]
    fig.add_trace(go.Bar(x=[b[0] for b in bars], y=[b[1] for b in bars], marker_color=[b[2] for b in bars],
                         text=[f"{b[1]:.3f}" for b in bars], textposition="outside", showlegend=False), 2, 1)
    lo = min(b[1] for b in bars)
    fig.update_yaxes(range=[lo - abs(lo) * 0.05 - 0.02, max(b[1] for b in bars) + 0.03], row=2, col=1)
    fig.add_trace(go.Bar(x=tr.seconds, y=tr.step, orientation="h", marker_color=TEAL, text=tr.decision, textposition="auto", showlegend=False), 2, 2)
    fig.update_yaxes(autorange="reversed", row=2, col=2)
    verdict = f"PROMOTED {agent.challenger}" if s.get("promoted") else f"KEPT {agent.champion}"
    fig.update_layout(template="plotly_white", height=820, barmode="overlay", legend=dict(orientation="h", y=1.08),
                      title=f"Retraining agent · outcome: {verdict}")
    fig.show()
    display(tr[["step", "finding", "decision", "seconds"]].style.hide(axis="index").set_caption("Agent decision log (auditable)"))

In [ ]:
# 🧾 Run log for this session
display(pd.DataFrame(RUNS))
with tolerant_drop():
    display(project.models()[["model_name", "status", "tabular_model_type", "instance_type", "created_at"]]
            .sort_values("created_at", ascending=False).head(20))

## Summary

| Capability | Where it appeared |
|---|---|
| Leakage guard before any training | section 3 |
| Zero-shot regression with foundation models | section 4 |
| Platform metrics and regression diagnostics | section 5 |
| Global SHAP and LIME, cross-model view | section 6 |
| Batch scoring: predicted vs actual, error bands | section 7 |
| Per-applicant explanation | section 8 |
| Ensemble, distillation to a fast student, leaderboard | section 9 |

**Notes for the presenter**
* An R² near 1.0 on this data means a leak, not a win: section 3 removes the fields that restate the loan amount.
* SHAP for regression is available for TabICLv2, XRFM, Limix, TabDPT, TabPFN v2.6/v3; Mitra, ContextTab, TabFM and ILTM are explained with LIME only.
* Support-set attribution is a classification feature; see the Lending Club notebook.